In [1]:
from pathlib import Path
import re

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import rasterio

from PIL import Image
from rasterio.enums import ColorInterp
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from shapely.geometry import box

In [2]:
# ============================================================
# 1. INPUT AND OUTPUT DIRECTORIES
# ============================================================

DATA_DIR = Path(r"D:\Data Satelit\Ruang\2class")

OUTPUT_DIR = DATA_DIR / "label"
MASK_TIF_DIR = OUTPUT_DIR / "mask_tif"
MASK_PNG_DIR = OUTPUT_DIR / "mask_png"
COLOR_TIF_DIR = OUTPUT_DIR / "color_tif"
COLOR_PNG_DIR = OUTPUT_DIR / "color_png"

for folder in [
    MASK_TIF_DIR,
    MASK_PNG_DIR,
    COLOR_TIF_DIR,
    COLOR_PNG_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# ============================================================
# 2. LEWOTOBI DOMAIN AND OUTPUT SIZE
# ============================================================

LAT_MIN = 0
LAT_MAX = 4.6
LON_MIN = 122.0
LON_MAX = 126.65

WIDTH = 256
HEIGHT = 256

TARGET_CRS = "EPSG:4326"


# ============================================================
# 3. CLASS VALUES AND COLORS
# ============================================================

# Single-band class mask
NON_ASH_CLASS = 0
ASH_CLASS = 1

# RGB visualization colors
NON_ASH_COLOR = np.array([0, 0, 255], dtype=np.uint8)  # Blue
ASH_COLOR = np.array([255, 0, 0], dtype=np.uint8)      # Red


# ============================================================
# 4. RASTERIZATION OPTION
# ============================================================

# False:
# A pixel is classified as ash only when its center falls
# inside the polygon.
#
# True:
# Every pixel touched by the polygon is classified as ash.
ALL_TOUCHED = False


In [3]:
# ============================================================
# 5. CREATE THE OUTPUT GRID
# ============================================================

transform = from_bounds(
    west=LON_MIN,
    south=LAT_MIN,
    east=LON_MAX,
    north=LAT_MAX,
    width=WIDTH,
    height=HEIGHT,
)

domain_polygon = box(
    LON_MIN,
    LAT_MIN,
    LON_MAX,
    LAT_MAX,
)

pixel_width_deg = (LON_MAX - LON_MIN) / WIDTH
pixel_height_deg = (LAT_MAX - LAT_MIN) / HEIGHT

print("=" * 70)
print("OUTPUT CONFIGURATION")
print("=" * 70)
print(f"Input folder       : {DATA_DIR}")
print(f"Output folder      : {OUTPUT_DIR}")
print(f"Output size        : {WIDTH} × {HEIGHT} pixels")
print(f"Longitude resolution: {pixel_width_deg:.8f} degrees/pixel")
print(f"Latitude resolution : {pixel_height_deg:.8f} degrees/pixel")
print(f"Longitude range    : {LON_MIN} to {LON_MAX}")
print(f"Latitude range     : {LAT_MIN} to {LAT_MAX}")
print()


# ============================================================
# 6. EXTRACT TIMESTAMP FROM FILENAME
# ============================================================

def extract_timestamp(file_stem: str) -> str:
    """
    Extract YYYYMMDD_HHMM from the filename.

    Example:
    Label_Lewotobi_20250801_1500
    becomes:
    20250801_1500
    """
    match = re.search(r"(\d{8}_\d{4})", file_stem)

    if match:
        return match.group(1)

    return file_stem

OUTPUT CONFIGURATION
Input folder       : D:\Data Satelit\Ruang\2class
Output folder      : D:\Data Satelit\Ruang\2class\label
Output size        : 256 × 256 pixels
Longitude resolution: 0.01816406 degrees/pixel
Latitude resolution : 0.01796875 degrees/pixel
Longitude range    : 122.0 to 126.65
Latitude range     : 0 to 4.6



In [4]:
# ============================================================
# 7. CHECK SHAPEFILE COMPONENTS
# ============================================================

def check_shapefile_components(shp_path: Path):
    """
    Check the main shapefile components.
    """
    base = shp_path.with_suffix("")

    required_extensions = [".shp", ".shx", ".dbf"]
    optional_extensions = [".prj", ".cpg"]

    missing_required = []
    missing_optional = []

    for extension in required_extensions:
        component = base.with_suffix(extension)

        if not component.exists():
            missing_required.append(component.name)

    for extension in optional_extensions:
        component = base.with_suffix(extension)

        if not component.exists():
            missing_optional.append(component.name)

    return missing_required, missing_optional

In [5]:
# ============================================================
# 8. READ AND PREPARE EACH SHAPEFILE
# ============================================================

def read_and_prepare_shapefile(
    shp_path: Path,
) -> gpd.GeoDataFrame:
    """
    Read the shapefile, convert it to EPSG:4326,
    repair invalid geometries, and clip it to the
    Lewotobi domain.
    """

    gdf = gpd.read_file(shp_path)

    if gdf.empty:
        raise ValueError("The shapefile is empty.")

    # Remove null or empty geometries
    gdf = gdf[
        gdf.geometry.notnull()
        & ~gdf.geometry.is_empty
    ].copy()

    if gdf.empty:
        raise ValueError(
            "No usable geometries were found."
        )

    # CRS is required for correct polygon positioning
    if gdf.crs is None:
        raise ValueError(
            "The CRS was not found. Make sure the .prj file "
            "exists and has the same base filename as the .shp file."
        )

    # Convert to WGS84
    if gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(TARGET_CRS)

    # Repair invalid geometries
    invalid = ~gdf.geometry.is_valid

    if invalid.any():
        gdf.loc[invalid, "geometry"] = (
            gdf.loc[invalid, "geometry"].buffer(0)
        )

    # Remove invalid or empty geometries after repair
    gdf = gdf[
        gdf.geometry.notnull()
        & ~gdf.geometry.is_empty
        & gdf.geometry.is_valid
    ].copy()

    if gdf.empty:
        raise ValueError(
            "No valid geometries remained after repair."
        )

    # Clip polygons to the Lewotobi domain
    gdf["geometry"] = (
        gdf.geometry.intersection(domain_polygon)
    )

    gdf = gdf[
        gdf.geometry.notnull()
        & ~gdf.geometry.is_empty
    ].copy()

    return gdf

In [6]:

# ============================================================
# 9. CREATE A SINGLE-BAND CLASS MASK
# ============================================================

def create_class_mask(
    gdf: gpd.GeoDataFrame,
) -> np.ndarray:
    """
    Create a class mask:

    0 = non-ash
    1 = volcanic ash
    """

    if gdf.empty:
        # If there is no ash polygon inside the domain,
        # create a full non-ash mask.
        return np.zeros(
            (HEIGHT, WIDTH),
            dtype=np.uint8,
        )

    shapes = [
        (geometry, ASH_CLASS)
        for geometry in gdf.geometry
        if geometry is not None
        and not geometry.is_empty
    ]

    mask = rasterize(
        shapes=shapes,
        out_shape=(HEIGHT, WIDTH),
        transform=transform,
        fill=NON_ASH_CLASS,
        dtype=np.uint8,
        all_touched=ALL_TOUCHED,
    )

    return mask


# ============================================================
# 10. CREATE THE RED-BLUE RGB MASK
# ============================================================

def create_rgb_mask(
    class_mask: np.ndarray,
) -> np.ndarray:
    """
    Convert the class mask to RGB:

    Blue = non-ash
    Red  = volcanic ash
    """

    rgb = np.zeros(
        (HEIGHT, WIDTH, 3),
        dtype=np.uint8,
    )

    rgb[class_mask == NON_ASH_CLASS] = NON_ASH_COLOR
    rgb[class_mask == ASH_CLASS] = ASH_COLOR

    return rgb


In [7]:
# ============================================================
# 11. SAVE THE SINGLE-BAND GEOTIFF
# ============================================================

def save_class_geotiff(
    output_path: Path,
    class_mask: np.ndarray,
    source_name: str,
    timestamp: str,
):
    """
    Save the class mask as a one-band GeoTIFF.

    0 = non-ash
    1 = volcanic ash
    """

    with rasterio.open(
        output_path,
        mode="w",
        driver="GTiff",
        height=HEIGHT,
        width=WIDTH,
        count=1,
        dtype=rasterio.uint8,
        crs=TARGET_CRS,
        transform=transform,
        compress="LZW",
    ) as dst:

        dst.write(class_mask, 1)

        dst.set_band_description(
            1,
            "Volcanic ash class mask",
        )

        dst.update_tags(
            source_shapefile=source_name,
            timestamp=timestamp,
            volcano="Lewotobi",
            class_0="non_ash",
            class_1="volcanic_ash",
            lon_min=LON_MIN,
            lon_max=LON_MAX,
            lat_min=LAT_MIN,
            lat_max=LAT_MAX,
            raster_width=WIDTH,
            raster_height=HEIGHT,
            all_touched=str(ALL_TOUCHED),
        )


# ============================================================
# 12. SAVE THE RGB GEOTIFF
# ============================================================

def save_rgb_geotiff(
    output_path: Path,
    rgb_mask: np.ndarray,
    source_name: str,
    timestamp: str,
):
    """
    Save the RGB visualization GeoTIFF.

    Red  = volcanic ash
    Blue = non-ash
    """

    with rasterio.open(
        output_path,
        mode="w",
        driver="GTiff",
        height=HEIGHT,
        width=WIDTH,
        count=3,
        dtype=rasterio.uint8,
        crs=TARGET_CRS,
        transform=transform,
        compress="LZW",
        photometric="RGB",
    ) as dst:

        dst.write(rgb_mask[:, :, 0], 1)
        dst.write(rgb_mask[:, :, 1], 2)
        dst.write(rgb_mask[:, :, 2], 3)

        dst.colorinterp = (
            ColorInterp.red,
            ColorInterp.green,
            ColorInterp.blue,
        )

        dst.set_band_description(1, "Red")
        dst.set_band_description(2, "Green")
        dst.set_band_description(3, "Blue")

        dst.update_tags(
            source_shapefile=source_name,
            timestamp=timestamp,
            volcano="Lewotobi",
            volcanic_ash_color="255,0,0",
            non_ash_color="0,0,255",
            lon_min=LON_MIN,
            lon_max=LON_MAX,
            lat_min=LAT_MIN,
            lat_max=LAT_MAX,
            raster_width=WIDTH,
            raster_height=HEIGHT,
        )

In [8]:
# ============================================================
# 13. FIND ALL LEWOTOBI SHAPEFILES
# ============================================================

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Input folder not found:\n{DATA_DIR}"
    )

shp_files = sorted(
    DATA_DIR.glob("Label_*.shp")
)

if not shp_files:
    raise FileNotFoundError(
        "No files matching Lewotobi_*.shp "
        f"were found in:\n{DATA_DIR}"
    )

print(f"Found {len(shp_files)} shapefiles.")
print()


# ============================================================
# 14. PROCESS ALL TIMESTAMPS
# ============================================================

success_records = []
failed_records = []

for index, shp_path in enumerate(
    shp_files,
    start=1,
):

    print("=" * 70)
    print(
        f"[{index}/{len(shp_files)}] Processing: "
        f"{shp_path.name}"
    )

    try:
        missing_required, missing_optional = (
            check_shapefile_components(shp_path)
        )

        if missing_required:
            raise FileNotFoundError(
                "Required shapefile components are missing: "
                + ", ".join(missing_required)
            )

        if missing_optional:
            print(
                "Warning: optional components are missing:",
                ", ".join(missing_optional),
            )

        file_stem = shp_path.stem
        timestamp = extract_timestamp(file_stem)

        class_tif_path = (
            MASK_TIF_DIR
            / f"{file_stem}_mask_256.tif"
        )

        class_png_path = (
            MASK_PNG_DIR
            / f"{file_stem}_mask_256.png"
        )

        color_tif_path = (
            COLOR_TIF_DIR
            / f"{file_stem}_color_256.tif"
        )

        color_png_path = (
            COLOR_PNG_DIR
            / f"{file_stem}_color_256.png"
        )

        # Read and prepare polygons
        gdf_clipped = read_and_prepare_shapefile(
            shp_path
        )

        # Create class and RGB masks
        class_mask = create_class_mask(
            gdf_clipped
        )

        rgb_mask = create_rgb_mask(
            class_mask
        )

        # Save class GeoTIFF
        save_class_geotiff(
            output_path=class_tif_path,
            class_mask=class_mask,
            source_name=shp_path.name,
            timestamp=timestamp,
        )

        # Save class PNG
        Image.fromarray(
            class_mask
        ).save(class_png_path)

        # Save RGB GeoTIFF
        save_rgb_geotiff(
            output_path=color_tif_path,
            rgb_mask=rgb_mask,
            source_name=shp_path.name,
            timestamp=timestamp,
        )

        # Save RGB PNG
        Image.fromarray(
            rgb_mask
        ).save(color_png_path)

        ash_pixels = int(
            np.sum(class_mask == ASH_CLASS)
        )

        non_ash_pixels = int(
            np.sum(class_mask == NON_ASH_CLASS)
        )

        ash_percentage = (
            ash_pixels
            / class_mask.size
            * 100.0
        )

        print(f"Timestamp          : {timestamp}")
        print(f"Polygons in domain : {len(gdf_clipped)}")
        print(f"Ash pixels         : {ash_pixels}")
        print(f"Non-ash pixels     : {non_ash_pixels}")
        print(f"Ash percentage     : {ash_percentage:.2f}%")
        print("Status             : SUCCESS")

        success_records.append(
            {
                "file": shp_path.name,
                "timestamp": timestamp,
                "polygon_count": len(gdf_clipped),
                "ash_pixels": ash_pixels,
                "non_ash_pixels": non_ash_pixels,
                "ash_percentage": ash_percentage,
            }
        )

    except Exception as error:
        print("Status             : FAILED")
        print("Reason             :", error)

        failed_records.append(
            {
                "file": shp_path.name,
                "error": str(error),
            }
        )

Found 14 shapefiles.

[1/14] Processing: Label_Lewotobi_20240417_1330.shp
Timestamp          : 20240417_1330
Polygons in domain : 2
Ash pixels         : 285
Non-ash pixels     : 65251
Ash percentage     : 0.43%
Status             : SUCCESS
[2/14] Processing: Label_Lewotobi_20240417_1400.shp
Timestamp          : 20240417_1400
Polygons in domain : 1
Ash pixels         : 1001
Non-ash pixels     : 64535
Ash percentage     : 1.53%
Status             : SUCCESS
[3/14] Processing: Label_Lewotobi_20240417_1430.shp
Timestamp          : 20240417_1430
Polygons in domain : 4
Ash pixels         : 1765
Non-ash pixels     : 63771
Ash percentage     : 2.69%
Status             : SUCCESS
[4/14] Processing: Label_Lewotobi_20240417_1500.shp
Timestamp          : 20240417_1500
Polygons in domain : 1
Ash pixels         : 2964
Non-ash pixels     : 62572
Ash percentage     : 4.52%
Status             : SUCCESS
[5/14] Processing: Label_Lewotobi_20240417_1530.shp
Timestamp          : 20240417_1530
Polygons in doma

In [9]:

# ============================================================
# 15. PROCESSING SUMMARY
# ============================================================

print()
print("=" * 70)
print("PROCESSING SUMMARY")
print("=" * 70)
print(f"Total input files  : {len(shp_files)}")
print(f"Successful         : {len(success_records)}")
print(f"Failed             : {len(failed_records)}")
print(f"Output folder      : {OUTPUT_DIR}")

if failed_records:
    print()
    print("Failed files:")

    for record in failed_records:
        print(
            f"- {record['file']}: "
            f"{record['error']}"
        )

print()
print("PROCESSING COMPLETE")


PROCESSING SUMMARY
Total input files  : 14
Successful         : 14
Failed             : 0
Output folder      : D:\Data Satelit\Ruang\2class\label

PROCESSING COMPLETE
